In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.neighbors import KNeighborsClassifier


## load and split iris dataset

In [ ]:
iris = datasets.load_iris()

X = iris.data
y = iris.target

print("Dataset shape:", X.shape)
print("Labels:", np.unique(y))
print("Class names:", iris.target_names)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("Training set size:", X_train.shape[0])
print("Test set size:", X_test.shape[0])


## Task 1 - Euclidean Distance

Euclidean distance measures how far apart two data points are.
KNN uses this distance to determine which training samples are
closest to a test sample.

In [ ]:

def Euclidean_distance(x1, x2):
    return np.sqrt(np.sum(np.square(x1 - x2)))


# Test the function
distance = Euclidean_distance(
    np.array([1, 2]),
    np.array([4, 6])
)

print("Distance:", distance)


## Task 2 - KNN From Scratch

The KNN class stores the training data using fit(). When predict()
is called, the model calculates the distance between each test sample
and the training samples.

The indexes of the k closest samples are found, and those indexes are
used to retrieve their class labels from y_train. The most common class
among the neighbors becomes the prediction.

In [ ]:
class KNN:
    def __init__(self,k):
        self.k = k

    def fit(self,X_train,y_train):
        self.X_train = X_train
        self.y_train = y_train

    def predict(self,X_test):
        predictions = []
        for x in X_test:
            predict = self.most_common_label(x)
            predictions.append(predict)
        return np.array(predictions)

    def most_common_label(self,x):
        distances = []
        for x_train in self.X_train:
            distance = Euclidean_distance(x, x_train)
            distances.append(distance)

        k_indices = np.argsort(distances)[:self.k]

        k_nearest_labels = [self.y_train[i] for i in k_indices]


        most_common_label = Counter(k_nearest_labels).most_common(1)[0][0]


        return most_common_label

## Task 3: Implement KNN

In [ ]:
k_values = [1, 3, 5, 7]

my_accuracies = []
my_confusion_matrices = []
my_predictions = {}

for k in k_values:

    model = KNN(k)

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    accuracy = accuracy_score(y_test, predictions)

    cm = confusion_matrix(y_test, predictions)

    my_accuracies.append(accuracy)
    my_confusion_matrices.append(cm)
    my_predictions[k] = predictions

    print("K =", k)
    print("Accuracy:", accuracy)
    print("Confusion Matrix:")
    print(cm)
    print()

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

for k in k_values:

    ConfusionMatrixDisplay.from_predictions(
        y_test,
        my_predictions[k],
        display_labels=iris.target_names
    )

    plt.title("My KNN Confusion Matrix - K=" + str(k))
    plt.show()


## Task 4: Compare with scikit-learn

In [ ]:
sklearn_accuracies = []
sklearn_confusion_matrices = []
sklearn_predictions_by_k = {}

for k in k_values:

    sklearn_model = KNeighborsClassifier(
        n_neighbors=k
    )

    sklearn_model.fit(
        X_train,
        y_train
    )

    sklearn_predictions = sklearn_model.predict(
        X_test
    )

    sklearn_accuracy = accuracy_score(
        y_test,
        sklearn_predictions
    )

    sklearn_cm = confusion_matrix(
        y_test,
        sklearn_predictions
    )

    sklearn_accuracies.append(sklearn_accuracy)
    sklearn_confusion_matrices.append(sklearn_cm)
    sklearn_predictions_by_k[k] = sklearn_predictions

    print("K =", k)
    print("Scikit-learn Accuracy:", sklearn_accuracy)
    print("Confusion Matrix:")
    print(sklearn_cm)
    print()

##Task 5: Plot and Analyze Results

In [ ]:
plt.plot(
    k_values,
    my_accuracies,
    marker="o",
    label="My KNN"
)

plt.plot(
    k_values,
    sklearn_accuracies,
    marker="o",
    label="Scikit-learn"
)

plt.xlabel("K Value")
plt.ylabel("Accuracy")
plt.title("KNN Accuracy vs K")

plt.xticks(k_values)
plt.ylim(0, 1.05)

plt.legend()

plt.show()

In [ ]:
print("K\tMy KNN\t\tScikit-learn")

for i in range(len(k_values)):
    print(
        k_values[i],
        "\t",
        round(my_accuracies[i], 4),
        "\t\t",
        round(sklearn_accuracies[i], 4)
    )

## Bias-Variance Tradeoff

The value of k controls how many neighboring training samples are used
to make a prediction.

A small value of k, such as k=1, makes the model very sensitive to
individual training samples. This gives the model lower bias but higher
variance. A small k can cause overfitting because one unusual training
sample can strongly affect a prediction.

As k increases, more neighboring samples vote on the classification.
This makes the model less sensitive to individual samples and lowers
variance. However, using too large of a k can increase bias because the
model begins considering points that are farther away from the test
sample. This can cause underfitting.

The goal is to choose a value of k that balances bias and variance.